## Hand written digit recognition
### PS : Hand written digit recognition

### Import required packages

In [1]:
import numpy as np
import torch
import torch.nn as nn

### use the required device

In [2]:
device=""
if torch.cuda.is_available():
    device = "cuda"
else:
    device="cpu"
print(f"using device : {device}")

using device : cuda


### Get the dataset

In [3]:
from torchvision import datasets
from torchvision import transforms
# create the transformer to transform the data into tensor
transformer =  transforms.ToTensor()

In [4]:

# Dwndload the training data 
train_data = datasets.MNIST(
    root = "./root",train=True ,download=True, transform=transformer  
)

In [5]:
# Dwndload the test data 
test_data = datasets.MNIST(
    root = "./root",train=False ,download=True, transform=transformer  
)

In [6]:
train_data

Dataset MNIST
    Number of datapoints: 60000
    Root location: ./root
    Split: Train
    StandardTransform
Transform: ToTensor()

In [7]:
test_data

Dataset MNIST
    Number of datapoints: 10000
    Root location: ./root
    Split: Test
    StandardTransform
Transform: ToTensor()

### Loader

In [8]:
from torch.utils.data import DataLoader

In [15]:
# Create a training Loader
train_loader = DataLoader(train_data,batch_size=64,shuffle=True)
test_loader  = DataLoader(test_data,batch_size=64,shuffle=True)

### Model defination

In [16]:
# create the model by adding the required layers 
# input for cnn 64 images 28*28 pixels in single batch 
# so at a time it recives the image of the suze of the 28*28
# input sixe 1

In [23]:
model = nn.Sequential(
    nn.Conv2d(in_channels=1,out_channels=16,kernel_size=3,padding=1,stride=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2,stride=2),

    #one more convulation layer
    nn.Conv2d(in_channels=16,out_channels=32,kernel_size=3,padding=1,stride=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2,stride=2),

    # Add the Flatten Layer
    nn.Flatten(),

    # Add the final output layer
    nn.Linear(in_features=32*7*7,out_features=128),
    nn.ReLU(),
    nn.Linear(in_features=128,out_features=10)
)
model = model.to(device)

In [24]:
model

Sequential(
  (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (1): ReLU()
  (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (4): ReLU()
  (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (6): Flatten(start_dim=1, end_dim=-1)
  (7): Linear(in_features=1568, out_features=128, bias=True)
  (8): ReLU()
  (9): Linear(in_features=128, out_features=10, bias=True)
)

### Define the Hypreparameters

In [27]:
epochs=5
loss_function = nn.CrossEntropyLoss()
learning_rate = 0.001
optimizer = torch.optim.Adam(model.parameters(),lr=learning_rate)

### Training Loop

In [30]:
losses = []
for epoch in range(epochs):
    model.train()
    running_loss=0
    for images,labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        
        pred = model(images)

        loss = loss_function(pred,labels)

        loss.backward()

        optimizer.step()

        running_loss+=loss.item()
    print(f"epoch = {epoch} , error = {running_loss}")

epoch = 0 , error = 231.77985206618905
epoch = 1 , error = 61.37613348197192
epoch = 2 , error = 42.93960440403316
epoch = 3 , error = 32.16253554925788
epoch = 4 , error = 26.30531827005325


### Evaluate the Model

In [31]:
model.eval()

Sequential(
  (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (1): ReLU()
  (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (4): ReLU()
  (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (6): Flatten(start_dim=1, end_dim=-1)
  (7): Linear(in_features=1568, out_features=128, bias=True)
  (8): ReLU()
  (9): Linear(in_features=128, out_features=10, bias=True)
)

In [32]:
# Define the stats 
correct = 0
total = 0

In [38]:
# disable the updating the weights
with torch.no_grad():
    for images,labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        pridictions = model(images)

        classes = torch.argmax(pridictions,dim=1)

        total+=labels.size(0)

        correct +=(classes == labels).sum().item()

accuracy = correct /total
print(accuracy)

0.9709576138147566


### Save the Model

In [39]:
torch.save(model.state_dict(),"mnist_model.pth")

### Test Model

In [48]:
image,label=test_data[0]
print(f"label for 1st test data = {label}")

label for 1st test data = 7


In [49]:
print(image.shape)

torch.Size([1, 28, 28])


In [53]:
image = image.to(device)
prediction = model(image)
print(prediction)

RuntimeError: mat1 and mat2 shapes cannot be multiplied (32x49 and 1568x128)

### how softmax works

In [33]:
numbers = [1,2,3,10,4,8,9,4]

np.argmax(numbers)

np.int64(3)